In [ ]:
# Google Colab : télécharger les données de cette séance / download this session's data
import os, urllib.request
BASE = "https://raw.githubusercontent.com/Marzoug-Nabil/Nabil/main/academic/teaching/fsci6222/materials/tp2/"
for f in ['diabete.csv']:
    os.makedirs('data', exist_ok=True)
    for dst in (f, os.path.join('data', f)):
        if not os.path.exists(dst):
            urllib.request.urlretrieve(BASE + f, dst)


# FSCI 6222 -- Analyse de donnees
# Travail pratique no 2 -- CORRIGE
## Analyse predictive du diabete

**Universite de Moncton -- Faculte des sciences -- Ete 2026**

---

Corrige complet du TP2 avec code et interpretations. Bareme : 50 points repartis sur 4 sections.


## Importations


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import (r2_score, mean_squared_error,
                              accuracy_score, confusion_matrix)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

---

# Section 1 -- Exploration et probabilites (10 points)


## 1.1 Chargement et nettoyage (3 points)


**Q1.** Charger et explorer. *(1 pt)*


In [ ]:
df = pd.read_csv('diabete.csv')
print('Dimensions :', df.shape)
print('\nTypes :')
print(df.dtypes)
df.head()

**Q2.** Valeurs manquantes + doublons. *(1 pt)*


In [ ]:
print('Valeurs manquantes par colonne :')
print(df.isnull().sum())
print(f'\nNombre de doublons : {df.duplicated().sum()}')

**Resultats attendus :** ~4 NaN dans `insuline`, ~4 NaN dans `pli_cutane`, ~4 doublons.


**Q3.** Nettoyage. *(1 pt)*


In [ ]:
df_clean = df.drop_duplicates().copy()

for col in df_clean.select_dtypes(include=[np.number]).columns:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(f'Dimensions apres nettoyage : {df_clean.shape}')
print(f'Valeurs manquantes restantes : {df_clean.isnull().sum().sum()}')

**Resultat attendu :** dimensions (400, 10), 0 valeurs manquantes.


## 1.2 Distribution de la glycemie (4 points)


**Q4.** Histogramme + moyenne + ecart-type de `glycemie`. *(1,5 pt)*


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_clean['glycemie'], bins=25, color='steelblue',
         edgecolor='white', alpha=0.8)
plt.xlabel('Glycemie a jeun (mg/dL)')
plt.ylabel('Frequence')
plt.title('Distribution de la glycemie')
plt.tight_layout()
plt.show()

mu = df_clean['glycemie'].mean()
sigma = df_clean['glycemie'].std()
print(f'Moyenne   : {mu:.2f} mg/dL')
print(f'Ecart-type: {sigma:.2f} mg/dL')

**Resultats attendus :** moyenne ~124, ecart-type ~37. L'histogramme presente une distribution bimodale (deux pics) car le dataset melange diabetiques et non-diabetiques.


**Q5.** P(glycemie > 140) sous loi normale. *(1,5 pt)*


In [ ]:
mu = df_clean['glycemie'].mean()
sigma = df_clean['glycemie'].std()

p_sup_140 = 1 - stats.norm.cdf(140, loc=mu, scale=sigma)

print(f'P(glycemie > 140) = {p_sup_140:.4f}')
print(f'Soit environ {p_sup_140*100:.1f}% des patientes')

**Resultat attendu :** P(glycemie > 140) ~ 0,34, soit environ 34 % des patientes auraient une glycemie superieure au seuil clinique sous l'hypothese de normalite.


**Q6.** Test de Shapiro-Wilk. *(1 pt)*


In [ ]:
stat, p_value = stats.shapiro(df_clean['glycemie'])
print(f'Statistique de test : {stat:.4f}')
print(f'p-value             : {p_value:.6f}')

**Resultats attendus :** stat ~ 0,946, p-value tres inferieure a 0,001.

**Interpretation :** la p-value est **bien inferieure a 0,05**, donc on **rejette** l'hypothese de normalite. La glycemie ne suit donc pas une loi normale sur cet echantillon. C'est coherent avec l'histogramme bimodal observe a la Q4 : le melange de patientes diabetiques et non-diabetiques cree deux pics, ce qui s'eloigne de la forme en cloche unique d'une distribution normale.


## 1.3 Probabilites discretes (3 points)


**Q7.** Proportion empirique de diabetiques. *(1 pt)*


In [ ]:
p_diab = df_clean['diabete'].mean()
print(f'Proportion de diabetiques : {p_diab:.4f}')
print(f'Soit environ {p_diab*100:.1f}% des patientes')

**Resultat attendu :** p ~ 0,44, soit environ 44 % des patientes sont diabetiques.


**Q8.** P(X >= 3) pour X ~ Binomiale(10, p). *(2 pt)*


In [ ]:
p_diab = df_clean['diabete'].mean()

# P(X >= 3) = 1 - P(X <= 2)
p_au_moins_3 = 1 - stats.binom.cdf(2, n=10, p=p_diab)

print(f'P(au moins 3 patientes diabetiques sur 10) = {p_au_moins_3:.4f}')
print(f'Soit environ {p_au_moins_3*100:.1f}%')

**Resultat attendu :** P(X >= 3) ~ 0,89, soit environ 89 %. Avec une prevalence de 44 % dans le dataset, il est tres probable qu'au moins 3 patientes sur 10 tirees au hasard soient diabetiques.


---

# Section 2 -- Correlation et regression (15 points)


## 2.1 Analyse des correlations (5 points)


**Q9.** Matrice de correlation + heatmap. *(2 pt)*


In [ ]:
cols_num = ['grossesses','glycemie','pression_arterielle','pli_cutane',
            'insuline','imc','heredite','age','diabete']
corr = df_clean[cols_num].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True)
plt.title('Matrice de correlation de Pearson')
plt.tight_layout()
plt.show()

**Q10.** Trois variables les plus correlees avec `diabete`. *(1,5 pt)*


In [ ]:
corr_diab = corr['diabete'].drop('diabete')
corr_triee = corr_diab.reindex(corr_diab.abs().sort_values(ascending=False).index)
print('Correlations avec diabete (triees par valeur absolue) :')
print(corr_triee)

**Resultats attendus :**

1. **`glycemie`** : r ~ +0,72 (correlation positive forte)
2. **`age`** : r ~ +0,57 (correlation positive moderee-forte)
3. **`imc`** : r ~ +0,50 (correlation positive moderee)

**Commentaire clinique :** ces trois correlations sont coherentes avec les connaissances medicales. Une glycemie elevee est par definition liee au diabete. L'age est un facteur de risque etabli (le diabete de type 2 apparait plus souvent apres 40 ans). L'IMC eleve est lie a l'obesite, qui est un facteur de risque majeur du diabete de type 2. Les correlations positives indiquent toutes que des valeurs elevees augmentent le risque.


**Q11.** Spearman vs Pearson sur `glycemie`. *(1,5 pt)*


In [ ]:
r_pearson, _ = stats.pearsonr(df_clean['glycemie'], df_clean['diabete'])
r_spearman, _ = stats.spearmanr(df_clean['glycemie'], df_clean['diabete'])

print(f'Pearson  : r = {r_pearson:.4f}')
print(f'Spearman : r = {r_spearman:.4f}')

**Resultats attendus :** Pearson ~ 0,72, Spearman ~ 0,73.

**Interpretation :** les deux coefficients sont tres proches, donc la relation entre glycemie et diabete est **approximativement lineaire**. Si Spearman avait ete nettement superieur, cela aurait suggere une relation monotone non lineaire (par exemple, un seuil au-dela duquel le risque s'envole brutalement).


## 2.2 Regression lineaire (10 points)


**Q12.** Regression simple. *(2 pt)*


In [ ]:
X1 = df_clean[['glycemie']]
y = df_clean['diabete']

modele_simple = LinearRegression()
modele_simple.fit(X1, y)

print(f"Ordonnee a l'origine : {modele_simple.intercept_:.4f}")
print(f'Pente               : {modele_simple.coef_[0]:.5f}')
print(f'R2                  : {modele_simple.score(X1, y):.4f}')

**Resultats attendus :** intercept ~ -0,76, pente ~ +0,0097, R2 ~ 0,52.

**Interpretation de la pente :** chaque augmentation d'1 mg/dL de glycemie augmente d'environ **0,0097** la probabilite estimee d'etre diabetique. Pour rendre l'effet plus parlant : **+10 mg/dL** augmentent la probabilite estimee d'environ **+10 %**.

**Note importante :** comme la cible `diabete` est binaire (0/1), la regression lineaire n'est pas le meilleur outil ici -- elle peut predire des valeurs en dehors de [0, 1]. La regression logistique serait plus appropriee. Mais l'exercice illustre que meme avec ce modele simple, la glycemie permet une bonne prediction lineaire.


**Q13.** Regression multiple. *(2 pt)*


In [ ]:
features = ['grossesses','glycemie','pression_arterielle','pli_cutane',
            'insuline','imc','heredite','age']
X = df_clean[features]
y = df_clean['diabete']

modele_multi = LinearRegression()
modele_multi.fit(X, y)

print(f"Ordonnee a l'origine : {modele_multi.intercept_:.4f}")
print('Coefficients :')
for nom, coef in zip(features, modele_multi.coef_):
    print(f'  {nom:22}: {coef:+.5f}')

**Resultats attendus :**
- intercept ~ -2,02
- glycemie : +0,0055 (effet principal)
- imc : +0,015
- pression_arterielle : +0,006
- age : +0,008
- heredite : +0,10
- les autres : effets plus faibles mais positifs

Tous les coefficients sont **positifs** : aucune variable ne diminue le risque selon le modele.


**Q14.** Train/test split + R2 et RMSE test. *(2,5 pt)*


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

modele = LinearRegression()
modele.fit(X_train, y_train)

y_pred = modele.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R2 test   = {r2:.4f}')
print(f'RMSE test = {rmse:.4f}')

**Resultats attendus :** R2 test ~ 0,75, RMSE ~ 0,25. Le modele multiple explique environ 75 % de la variance, c'est tres bon.


**Q15.** Diagramme des residus. *(2 pt)*


In [ ]:
y_pred_test = modele.predict(X_test)
residus = y_test - y_pred_test

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_test, residus, alpha=0.6, color='steelblue')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Valeurs predites')
plt.ylabel('Residus')
plt.title('Diagramme des residus -- regression multiple')
plt.tight_layout()
plt.show()

**Interpretation attendue :** comme la cible est **binaire**, les residus forment **deux bandes** (au-dessus et en dessous de zero). Ce n'est pas le diagramme attendu pour une regression sur cible continue. C'est un signal clair que la regression lineaire n'est pas optimale pour une cible binaire : il vaudrait mieux utiliser une regression logistique ou un classificateur (comme en section 4). Le diagramme illustre donc une limite importante de la regression lineaire dans ce contexte.


**Q16.** Comparaison R2 simple vs R2 multiple. *(1,5 pt)*


**Reponse :**

- R2 de la regression simple (Q12) ~ **0,52**
- R2 test de la regression multiple (Q14) ~ **0,75**

L'ajout des autres variables fait passer le R2 de 0,52 a 0,75 -- une amelioration tres substantielle (~+23 points). Les variables qui apportent le plus d'information complementaire sont : **`age`** (correlation 0,57), **`imc`** (correlation 0,50), **`pression_arterielle`** (0,48), **`insuline`** (0,49) et **`heredite`** (0,24). Chaque facteur de risque clinique apporte une information additionnelle au-dela de la seule glycemie.


---

# Section 3 -- Reduction de dimensionnalite (10 points)


## 3.1 Preparation des donnees (2 points)


**Q17.** Pourquoi standardiser. *(1 pt)*


**Reponse :** la PCA et le t-SNE reposent sur des **distances** et des **variances**. Si les variables ne sont pas mises a l'echelle, celles qui ont une grande amplitude (par exemple `insuline` qui varie de 10 a 600 mu U/mL) dominent celles qui ont une petite amplitude (par exemple `heredite` qui varie de 0,05 a 2,5). Standardiser (centrer et reduire) ramene toutes les variables sur la meme echelle (moyenne 0, ecart-type 1), ce qui permet de comparer leurs contributions de maniere equitable.


**Q18.** Application du StandardScaler. *(1 pt)*


In [ ]:
features = ['grossesses','glycemie','pression_arterielle','pli_cutane',
            'insuline','imc','heredite','age']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[features])

print(f'Forme de X_scaled : {X_scaled.shape}')
print(f'Moyennes (doivent etre ~ 0) : {X_scaled.mean(axis=0).round(4)}')
print(f'Ecarts-types (doivent etre ~ 1) : {X_scaled.std(axis=0).round(4)}')

## 3.2 PCA (5 points)


**Q19.** PCA 2 composantes + variance expliquee. *(2 pt)*


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'Variance expliquee PC1 : {pca.explained_variance_ratio_[0]:.4f}')
print(f'Variance expliquee PC2 : {pca.explained_variance_ratio_[1]:.4f}')
print(f'Total cumule           : {pca.explained_variance_ratio_.sum():.4f}')

**Resultats attendus :** PC1 ~ 34 %, PC2 ~ 15 %, total ~ 49 %. Les deux premieres composantes capturent environ la moitie de l'information totale.


**Q20.** Nuage PCA colore selon `diabete`. *(2 pt)*


In [ ]:
plt.figure(figsize=(9, 6))
couleurs = df_clean['diabete'].map({0: 'steelblue', 1: 'crimson'})
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=couleurs, alpha=0.6,
            edgecolor='k', linewidth=0.3)

# Legende manuelle
from matplotlib.lines import Line2D
legende = [Line2D([0],[0], marker='o', color='w', label='Non diabetique',
                   markerfacecolor='steelblue', markersize=8),
           Line2D([0],[0], marker='o', color='w', label='Diabetique',
                   markerfacecolor='crimson', markersize=8)]
plt.legend(handles=legende)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Projection PCA -- coloration selon le statut diabetique')
plt.tight_layout()
plt.show()

**Q21.** Commentaire sur les regroupements PCA. *(1 pt)*


**Reponse :** on observe une **separation visible** entre les patientes diabetiques et non diabetiques selon l'axe PC1. Les diabetiques (rouges) tendent a se regrouper a droite, les non-diabetiques (bleus) a gauche, avec une zone de chevauchement au milieu. PC1 capte donc l'essentiel des variables liees au diabete (glycemie, age, IMC, pression). Cette bonne separation visuelle suggere qu'un modele de classification simple devrait bien fonctionner -- ce qu'on confirmera a la section 4.


## 3.3 t-SNE (3 points)


**Q22.** t-SNE. *(2 pt)*


In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
couleurs = df_clean['diabete'].map({0: 'steelblue', 1: 'crimson'})
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=couleurs, alpha=0.6,
            edgecolor='k', linewidth=0.3)
legende = [Line2D([0],[0], marker='o', color='w', label='Non diabetique',
                   markerfacecolor='steelblue', markersize=8),
           Line2D([0],[0], marker='o', color='w', label='Diabetique',
                   markerfacecolor='crimson', markersize=8)]
plt.legend(handles=legende)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title('Projection t-SNE -- coloration selon le statut diabetique')
plt.tight_layout()
plt.show()

**Q23.** Comparaison PCA vs t-SNE. *(1 pt)*


**Reponse :** le **t-SNE** offre souvent une separation plus visible entre les deux groupes, parce qu'il est une methode **non lineaire** capable de capter des structures locales que la PCA (lineaire) manque. Sur ce dataset, les deux donnent une separation correcte, mais t-SNE tend a creer des regroupements plus compacts. Rappel important : t-SNE est utile pour la **visualisation** uniquement -- les distances dans son espace n'ont pas d'interpretation absolue et il ne permet pas de projeter de nouvelles donnees.


---

# Section 4 -- Apprentissage automatique (15 points)


## 4.1 Preparation du probleme de classification (3 points)


**Q24.** Distribution de la cible. *(1,5 pt)*


In [ ]:
print('Distribution de la cible diabete :')
print(df_clean['diabete'].value_counts())
print(f"\nProportion de diabetiques : {df_clean['diabete'].mean():.4f}")

**Resultat attendu :** environ 224 non-diabetiques (0) et 176 diabetiques (1), soit 56 % vs 44 %. La distribution est **legerement desequilibree**, mais reste assez equilibree pour la classification sans techniques particulieres.


**Q25.** Train/test split. *(1,5 pt)*


In [ ]:
y = df_clean['diabete']

Xs_train, Xs_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

print(f'Train : {Xs_train.shape[0]} patientes')
print(f'Test  : {Xs_test.shape[0]} patientes')

## 4.2 Classification : KNN vs Arbre de decision (8 points)


**Q26.** KNN avec k=5. *(2 pt)*


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(Xs_train, y_train)

y_pred_knn = knn.predict(Xs_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f'KNN accuracy : {acc_knn:.4f}')

**Resultat attendu :** accuracy ~ 0,95. Excellente performance.


**Q27.** Arbre avec max_depth=4. *(2 pt)*


In [ ]:
arbre = DecisionTreeClassifier(max_depth=4, random_state=42)
arbre.fit(Xs_train, y_train)

y_pred_arbre = arbre.predict(Xs_test)
acc_arbre = accuracy_score(y_test, y_pred_arbre)
print(f'Arbre accuracy : {acc_arbre:.4f}')

**Resultat attendu :** accuracy ~ 0,90. Tres bon mais le KNN fait mieux.


**Q28.** Matrices de confusion. *(2 pt)*


In [ ]:
labels = [0, 1]
labels_txt = ['Non diabetique', 'Diabetique']

cm_knn = confusion_matrix(y_test, y_pred_knn, labels=labels)
cm_arbre = confusion_matrix(y_test, y_pred_arbre, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_txt, yticklabels=labels_txt, ax=axes[0])
axes[0].set_title(f'KNN (accuracy = {acc_knn:.3f})')
axes[0].set_xlabel('Predit')
axes[0].set_ylabel('Reel')

sns.heatmap(cm_arbre, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_txt, yticklabels=labels_txt, ax=axes[1])
axes[1].set_title(f'Arbre (accuracy = {acc_arbre:.3f})')
axes[1].set_xlabel('Predit')
axes[1].set_ylabel('Reel')

plt.tight_layout()
plt.show()

**Q29.** Comparaison + reflexion clinique. *(2 pt)*


**Reponse :** le **KNN** obtient une meilleure accuracy (~0,95) que l'**arbre** (~0,90) sur ce dataset.

**Faux negatifs vs faux positifs en contexte medical :**
- Un **faux negatif** = declarer non-diabetique une personne qui l'est. **Tres grave** : la patiente n'est pas suivie, la maladie progresse sans traitement, complications cardiovasculaires, renales, oculaires peuvent s'installer.
- Un **faux positif** = declarer diabetique une personne saine. **Genant** mais moins grave : ca declenche des examens complementaires (test de tolerance au glucose, HbA1c) qui corrigeront le diagnostic rapidement.

**Conclusion clinique :** pour un test de **depistage**, on prefere minimiser les **faux negatifs**, quitte a accepter plus de faux positifs. Il faut donc regarder la **sensibilite** (rappel sur la classe diabetique) plutot que la seule accuracy. Dans la matrice de confusion, le modele qui a le plus petit nombre dans la cellule "Reel: Diabetique, Predit: Non diabetique" est le meilleur. Sur la base des matrices ci-dessus, c'est le modele qui doit etre privilegie pour le depistage.


## 4.3 Clustering : K-Means (4 points)


**Q30.** K-Means avec n_clusters=2. *(2 pt)*


In [ ]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
kmeans.fit(X_scaled)

df_clean['cluster'] = kmeans.labels_

print('Taille de chaque cluster :')
print(df_clean['cluster'].value_counts().sort_index())

**Resultat attendu :** deux clusters de tailles ~156 et ~244.


**Q31.** Crosstab clusters vs diabete. *(2 pt)*


In [ ]:
tab = pd.crosstab(df_clean['cluster'], df_clean['diabete'],
                   rownames=['Cluster'], colnames=['Diabete'])
print(tab)
print(f'\nProportion de diabetiques dans chaque cluster :')
for c in sorted(df_clean['cluster'].unique()):
    sous_df = df_clean[df_clean['cluster'] == c]
    print(f'  Cluster {c} : {sous_df["diabete"].mean()*100:.1f}% diabetiques')

**Resultats attendus :**

- Cluster 0 : ~156 patientes, dont **~97 % de diabetiques** (152 sur 156)
- Cluster 1 : ~244 patientes, dont **~90 % de non-diabetiques** (220 sur 244)

**Interpretation :** les clusters trouves par K-Means **correspondent tres bien** aux deux groupes de patientes. C'est remarquable : K-Means est un algorithme **non supervise**, il n'a pas vu la variable `diabete`. Pourtant, en se basant uniquement sur les similarites cliniques (glycemie, age, IMC, etc.), il decoupe presque parfaitement la population en diabetiques et non-diabetiques. Cela confirme que les huit variables cliniques portent une information forte et coherente sur le statut diabetique : les patientes diabetiques se ressemblent entre elles, et les non-diabetiques aussi, ce qui rend le diagnostic predictif possible.


---

**Fin du corrige TP2.**

**Bareme total : 50 points** repartis ainsi :

| Section | Points |
|---|---|
| 1. Exploration et probabilites | 10 |
| 2. Correlation et regression | 15 |
| 3. Reduction de dimensionnalite | 10 |
| 4. Apprentissage automatique | 15 |
| **Total** | **50** |
